**Install Required Packages**

In [1]:
!pip install pandas --quiet
!pip install tabulate --quiet
!pip install pdfplumber --quiet
!pip install pymupdf --upgrade --prefer-binary --only-binary :all:

**Define page-finder & table-extractor functions**

In [2]:
import pandas as pd

def find_pages(pdf_path: str, keyword: str) -> list[int]:
    """Return 1-based page numbers where `keyword` appears in the page text."""
    pages = []
    with fitz.open(pdf_path) as doc:
        for i in range(doc.page_count):
            if keyword in doc[i].get_text():
                pages.append(i + 1)
    return pages


def extract_period_tables(
    pdf_path: str,
    pages: list[int],
    period: str,
    drop_type: str = "Spread",
) -> pd.DataFrame:
    """
    From the given pages, pull out every table row where
    df['Period']==period AND df['Type']!=drop_type.
    Returns one concatenated DataFrame (or empty if none).
    """
    dfs = []
    with pdfplumber.open(pdf_path) as pdf:
        for pnum in pages:
            t = pdf.pages[pnum - 1].extract_table()
            if not t:
                continue
            df = pd.DataFrame(t[1:], columns=t[0])
            if "Type" in df.columns:
                df["Type"] = df["Type"].str.strip()
                df = df[df["Type"] != drop_type]
            if "Period" in df.columns:
                df = df[df["Period"] == period]
            if not df.empty:
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

**Run the pipeline & display**

1. Read in all the PDF files in the folder
2. Extract the tables with Daily contracts

In [3]:
from pathlib import Path
import pandas as pd

import fitz
import pdfplumber

pdf_folder  = Path(".")
target = "Daily"

all_dfs = []
for pdf_file in pdf_folder.glob("*.pdf"):
    pages = find_pages(str(pdf_file), target)
    df = extract_period_tables(str(pdf_file), pages, target)
    if not df.empty:
        all_dfs.append(df)

result = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

# final cleanup: unique Display Name + drop unwanted cols
result = (
    result
    .drop_duplicates(subset=["Display Name"], keep="first")
    .drop(columns=["Type", "Buyer", "Seller", "Exp Time"])
)

**Determine if In The Money**

1. Compare the Strike Price with the price in the description

In [4]:
import pandas as pd
from tabulate import tabulate

result = (
    result
    .assign(
        # coerce Exp Value from str → float
        **{'Exp Value': lambda df: pd.to_numeric(df['Exp Value'], errors='coerce')},
        # pull the number after '>' out of Display Name
        Threshold=lambda df: (
            df['Display Name']
              .str.extract(r'>\s*([\d.]+)', expand=False)
              .pipe(pd.to_numeric, errors='coerce')
        ),
        # compare and cast bool → int
        Flag=lambda df: (df['Exp Value'] > df['Threshold']).astype(int),
    )
    .drop(columns='Threshold')  # clean up
)
result.rename(columns={"Business Date": "Date", "Exp Value": "Strike", "Flag": "In the Money", },              inplace=True)

# show your results
if result.empty:
    print("No matching rows found.")
else:
    print(tabulate(result.tail(25),
                   headers="keys",
                   tablefmt="fancy_grid",
                   showindex=False))

╒═══════════╤══════════╤═════════════════╤═══════════════════════════════════╤══════════╤════════════════╕
│ Date      │ Period   │ Name            │ Display Name                      │   Strike │   In the Money │
╞═══════════╪══════════╪═════════════════╪═══════════════════════════════════╪══════════╪════════════════╡
│ 13-JUN-25 │ Daily    │ USTECH_JUN_25   │ US Tech 100 (Jun) >22228 (4:15PM) │  21677   │              0 │
├───────────┼──────────┼─────────────────┼───────────────────────────────────┼──────────┼────────────────┤
│ 13-JUN-25 │ Daily    │ USTECH_JUN_25   │ US Tech 100 (Jun) >22276 (4:15PM) │  21677   │              0 │
├───────────┼──────────┼─────────────────┼───────────────────────────────────┼──────────┼────────────────┤
│ 13-JUN-25 │ Daily    │ USTECH_JUN_25   │ US Tech 100 (Jun) >22324 (4:15PM) │  21677   │              0 │
├───────────┼──────────┼─────────────────┼───────────────────────────────────┼──────────┼────────────────┤
│ 13-JUN-25 │ Daily    │ USTECH_JUN_2

**Map the Name to a Ticker Symbol**

1. Use the ticker_from_description Comma Separated Value file (Update that if other tickers are desired)

In [5]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# 1) load your mapping table
pdf_folder   = Path(".")  # same folder as before
mapping_file = pdf_folder / "ticker_from_description.csv"

mapping_df = pd.read_csv(mapping_file)  # expect columns: Description, Ticker

# 2) define a helper that finds the first matching ticker
def lookup_ticker(name: str) -> str:
    for desc, ticker in zip(mapping_df['Display Name'], mapping_df['Ticker']):
        if pd.notna(name) and desc in name:
            return ticker
    return pd.NA

result['Ticker'] = pd.NA

for _, row in mapping_df.iterrows():
    mask = result['Name'].str.contains(row['Display Name'], na=False)
    result.loc[mask, 'Ticker'] = row['Ticker']
    
# show your results

result.drop(columns=["Name", "Period", "Display Name"], inplace=True)

print_order = [
    'Date',
    'Ticker',
    'Strike',
    'In the Money',
]
if result.empty:
    print("No matching rows found.")
else:
    print(tabulate(result[print_order].tail(30),
                   headers="keys",
                   tablefmt="fancy_grid",
                   showindex=False))

╒═══════════╤══════════╤══════════╤════════════════╕
│ Date      │ Ticker   │   Strike │   In the Money │
╞═══════════╪══════════╪══════════╪════════════════╡
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼────────────────┤
│ 13-JUN-25 │ NQ=F     │  21677   │              0 │
├───────────┼──────────┼──────────┼───────────